# HOUSING WORKBOOK - MACHINE LEARNING

### DATASET LINK  

*https://www.kaggle.com/datasets/vallabhadattap/kingcountyhousing*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 1. Get and understand the data

In [ ]:
df = pd.read_csv("KC_housing_data.csv")
print(df.shape)
print(df.head)

## 2. Explore and clean

In [ ]:
missing = df.isna().sum() # missing values in the dataset check
missing

In [ ]:
df["street"].describe()

In [ ]:
dups = df[df["street"].duplicated(keep=False)].sort_values(["street", "date"]) # duplicate houses on the dataset were houses that were sold multiple times
dups[["street", "date", "price", "sqft_living", "bedrooms"]].head(20)

In [ ]:
print("duplicate rows :" , df.duplicated().sum())

In [ ]:
df = df[df["price"] > 0] # no NaNs but some values can still be 0 which would skew the result

In [ ]:
(df["price"] == 0).sum()

## 3. Choosing X and Y variables

In [ ]:
features = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors", "condition"]
price = "price"

X = df[features]
Y = df[price]

## 4. Check initial regression assumptions

There are two main assumpions made about the data to check for initial regression.

### Linearity
We assume linearity in that as the x variable features increase, the y variable of price should also increase. For example, if the number of bedrooms offered in the house increases, its value naturally should increase.

### No multicollinearity
There were multiple different columns in this dataset that might have been correlated with each other in a way that would disrupt the model and destabilise the coefficients (the coefficients are essentially multipliers that measure the impact of one variable to another). An example of this are the multiple different square feet measurements of each floor. Conflicting columns are removed.

## 5. Split into training & test data

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 50)

## 6. Create and train the model

In [ ]:
model = LinearRegression()    # The negative bedrooms coefficient looks wrong but has a standard explanation. It’s the effect of adding a bedroom while holding sqft_living fixed. If total floor area doesn’t change, an extra bedroom means smaller rooms, which buyers value less. Bedrooms and living area are strongly correlated, so the model splits their effect in a way that looks odd. This is multicollinearity.
model.fit(X_train, Y_train)

print("Intercept:", model.intercept_)
pd.Series(model.coef_, index=features).sort_values()

## 7. Make predictions

In [ ]:
Y_pred = model.predict(X_test)
pd.DataFrame({"actual": Y_test, "predicted": Y_pred})

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(Y_test, Y_pred, s=8, alpha = 0.4)


lims = [min(Y_test.min(), Y_pred.min()), max(Y_test.max(), Y_pred.max())]
plt.plot(lims, lims, "r--", label="Perfect prediction") # line of best fit
plt.xlim(0, 2_000_000)
plt.ylim(0, 2_000_000)
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.title("Actual vs predicted house prices")

In [ ]:
check = X_test.copy()
check["actual"] = Y_test
check["predicted"] = Y_pred
check["error"] = check["actual"] - check["predicted"]
check = check.join(df[["city", "statezip"]])
check.sort_values("error", ascending=False).head(10)

## 8. Evaluate model performance

The model was succesful, and the line of best fit matched x = y practically perfectly, showing that the model learned the linearity patterns from the features vs the price. The main issue was that with less data on the higher value properties, the model clearly undervalues the prices of the costlier homes, and by a substantial amount. This may have been handled by the model due to it wanting to reduce error and not throw out higher end predictions but instead move them closer to the average value.

In [ ]:
mae = mean_absolute_error(Y_test, Y_pred)
rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))
r2 = r2_score(Y_test, Y_pred)

print(mae)
print(rmse)
print(r2)

## 9. Analyse residuals and check remaining assumptions

In [ ]:
residuals = Y_test - Y_pred
plt.scatter(Y_pred, residuals, s=6, alpha=0.4)
plt.axhline(0, color="r", linestyle="--")
plt.xlabel("Predicted price"); plt.ylabel("Residual"); plt.show()

In [ ]:
from scipy import stats
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(residuals, bins=50); ax[0].set_title("Residual distribution")
stats.probplot(residuals, plot=ax[1]); ax[1].set_title("Q-Q plot")
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for a, col in zip(ax.ravel(), features):
    a.scatter(X_test[col], residuals, s=5, alpha=0.3)
    a.axhline(0, color="r"); a.set_title(col)
plt.show()

In [ ]:
plt.hist(Y_test, bins=50, range=(0, 2_000_000), alpha=0.5, label="actual")
plt.hist(Y_pred, bins=50, range=(0, 2_000_000), alpha=0.5, label="predicted")
plt.legend();
plt.title("Variation in Actual vs Predicted price");
plt.xlabel("Price"); 
plt.show() 

The residuals seem to be a healthy and manageable amount. As for the remaining assumptions:

### Autocorrelation
The residuals should be independent from house to house, but in theory the economy or trends of the market might have affected all homes during this short period of data.

### Homoscedasticity
The variance of the residuals are not constant across all levels of the predictions. The more expensive the home, the higher the error in prediction vs actual value, which would be something to analyse further, potentially by adding more higher value property data could the model be made to be better.
